# 深度学习笔记02：神经网络和反向传播算法
> 学习材料：零基础入门深度学习(3)‑神经网络和反向传播算法
> 核心目标：理解全连接神经网络结构，掌握sigmoid激活函数；弄懂反向传播算法原理、误差项δ计算；阅读原版面向对象Node‑Layer‑Connection实现；理解链式求导；了解梯度检查手段。

## 1. 神经元与Sigmoid激活函数
普通神经元和感知器结构大体一致，区别是激活函数换成sigmoid。

sigmoid函数定义：
$
sigmoid(x)=\frac{1}{1+e^{-x}}
$
值域：$(0,1)$
输出可以解释成概率。

sigmoid导数：
令 $y = sigmoid(x)$
$
y' = y(1-y)
$

思考：
- 阶跃函数不可导，无法使用梯度下降；sigmoid处处可导，这是使用它的核心原因。
- sigmoid导数不需要重新做指数运算，直接使用自身输出 $y(1-y)$，代码计算开销很小。
- sigmoid输入绝对值很大的时候，导数趋近0，会出现**梯度消失**，深层网络训练会受影响。

神经元输出公式：
$
y = sigmoid(\boldsymbol{w}^T \boldsymbol{x} + b)
$

## 2. 全连接神经网络结构
全连接网络规则：
1. 神经元按层划分：输入层、隐藏层、输出层；输入层和输出层中间全部叫隐藏层。
2. **层内神经元之间没有连接**；
3. 第N层每一个神经元，和N‑1层**全部神经元相连**，这就是“全连接”；
4. 每条连接带有权重；偏置使用ConstNode（恒输出为1的节点）实现。

思考：
- 超参数：网络层数、每层节点数量，这些不是训练学出来的，需要人手动设置。
- 权重w是模型参数，训练过程更新；层数、节点数是超参数。
- 增加隐藏层（深度）可以提升模型表达能力，但是网络变深会带来梯度消失、训练难度上升。

## 3. 前向传播
前向传播：从输入层输入样本数据，一层一层计算，直到算出输出层结果。

流程：
1. 将样本赋值给输入层各个节点输出；
2. 对每一层的每一个神经元，用上一层所有节点输出 × 对应连接权重求和，经过sigmoid激活，得到本层节点输出；
3. 逐层向后计算，最后得到输出层输出向量。

思考：
- 前向传播只是做预测。网络训练真正的核心是反向传播：根据预测和真实标签的误差，从输出层往回算每一个节点的误差项δ，再更新所有权重。

## 4. 反向传播算法原理
目标：最小化输出层平方误差，使用随机梯度下降更新所有权重。

符号约定
- $t_i$：节点i真实标签
- $y_i$：节点i前向传播输出
- $\delta_i$：节点误差项（敏感度，损失对该节点加权输入的负偏导）

### 输出层节点误差项δ
$
\delta_i = y_i(1-y_i)(t_i - y_i)
$

### 隐藏层节点误差项δ
隐藏层没有标签，误差来自后面一层传回：
$
\delta_i = y_i(1-y_i)\sum_{k\in下游节点} \big(\delta_k \cdot w_{ki}\big)
$
$w_{ki}$：当前节点i到下游节点k的连接权重。

### 权重更新规则
$
w_{ji} \leftarrow w_{ji}+\eta \cdot \delta_j \cdot a_i
$
- $a_i$：上游节点i的输出
- $\delta_j$：下游节点j的误差项
- $\eta$：学习率

思考:
- 计算顺序必须**从输出层往输入层倒着算δ**，所以叫反向传播。要先拿到后一层的δ，才能算当前隐藏层δ。
- 公式全部基于链式求导法则。
- ConstNode（偏置节点）输出恒等于1，更新偏置本质和普通权重更新公式完全一样。

概念区分：
前向传播：输入→输出，算预测；
反向传播：输出误差反向回传，算各个节点δ，更新全部权重。

## 5. 代码介绍
Node类定义
功能：
1. 记录所属层编号、节点编号；保存上游、下游连接
2. calc_output：前向，计算本节点输出（sigmoid）
3. calc_output_layer_delta：输出层节点计算误差项δ
4. calc_hidden_layer_delta：隐藏层节点计算误差项δ
5. \_\_str\_\_：打印节点调试信息

In [1]:
from functools import reduce

class Node(object):
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0
        self.delta = 0

    def set_output(self, output):
        self.output = output

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def append_upstream_connection(self, conn):
        self.upstream.append(conn)

    def calc_output(self):
        output = reduce(lambda ret, conn: ret + conn.upstream_node.output * conn.weight, self.upstream, 0.0)
        self.output = 1.0/(1.0 + pow(2.71828, -output))

    def calc_hidden_layer_delta(self):
        downstream_delta = reduce(
            lambda ret, conn: ret + conn.downstream_node.delta * conn.weight,
            self.downstream, 0.0)
        self.delta = self.output * (1 - self.output) * downstream_delta

    def calc_output_layer_delta(self, label):
        self.delta = self.output * (1 - self.output) * (label - self.output)

    def __str__(self):
        node_str = '%u-%u: output: %f delta: %f' % (self.layer_index, self.node_index, self.output, self.delta)
        downstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn), self.downstream, '')
        upstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn), self.upstream, '')
        return node_str + '\n\tdownstream:' + downstream_str + '\n\tupstream:' + upstream_str


ConstNode偏置节点类
功能：
1. 专门实现bias，output恒等于1，不需要计算输出；
2. 只需要实现calc_hidden_layer_delta，参与反向传播误差计算；
3. 每一层Layer内部会自动创建一个ConstNode。

思考：
把bias当做一个输出固定为1的特殊节点，不需要单独写一套bias更新逻辑，复用Connection权重更新，简化代码结构。

In [2]:
class ConstNode(object):
    def __init__(self, layer_index, node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.output = 1

    def append_downstream_connection(self, conn):
        self.downstream.append(conn)

    def calc_hidden_layer_delta(self):
        downstream_delta = reduce(
            lambda ret, conn: ret + conn.downstream_node.delta * conn.weight,
            self.downstream, 0.0)
        self.delta = self.output * (1 - self.output) * downstream_delta

    def __str__(self):
        node_str = '%u-%u: output: 1' % (self.layer_index, self.node_index)
        downstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn), self.downstream, '')
        return node_str + '\n\tdownstream:' + downstream_str

Layer层类
功能：
1. 初始化一层：创建普通Node，末尾自动追加一个ConstNode（偏置节点）
2. set_output：给输入层设置样本输入
3. calc_output：遍历本层所有普通节点，执行前向输出计算
4. dump：打印层全部节点调试信息

思考：
每一层节点列表最后一个元素永远是ConstNode，计算输出的时候要把它跳过，不能对ConstNode调用calc_output。

In [3]:
class Layer(object):
    def __init__(self, layer_index, node_count):
        self.layer_index = layer_index
        self.nodes = []
        for i in range(node_count):
            self.nodes.append(Node(layer_index, i))
        self.nodes.append(ConstNode(layer_index, node_count))

    def set_output(self, data):
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        for node in self.nodes:
            print(node)

Connection连接类
每一条连接，连接上游节点、下游节点；保存权重weight，保存梯度gradient。
功能：
1. \_\_init\_\_：上下游节点，权重随机初始化（‑0.1 ~ 0.1）
2. calc_gradient：根据下游节点delta，计算该连接权重梯度
3. update_weight：使用学习率更新权重
4. \_\_str\_\_：打印连接信息

思考：
梯度 = 下游节点delta × 上游节点输出，对应反向传播权重更新公式。

In [4]:
import random
class Connection(object):
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node
        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = self.downstream_node.delta * self.upstream_node.output

    def get_gradient(self):
        return self.gradient

    def update_weight(self, rate):
        self.calc_gradient()
        self.weight += rate * self.gradient

    def __str__(self):
        return '(%u-%u) -> (%u-%u) = %f' % (
            self.upstream_node.layer_index,
            self.upstream_node.node_index,
            self.downstream_node.layer_index,
            self.downstream_node.node_index,
            self.weight)

class Connections(object):
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)

    def dump(self):
        for conn in self.connections:
            print(conn)

Network网络顶层类
参数layers：列表，描述网络每层节点数量，例如 [2,3,1] 代表输入层2个，隐藏层3个，输出层1个。
功能：
1. \_\_init\_\_：创建所有Layer，自动生成层与层之间全部Connection全连接；
2. train：训练入口，循环迭代，逐个样本调用train_one_sample（随机梯度下降SGD）
3. train_one_sample：单样本完整流程：predict前向 → calc_delta反向算误差 → update_weight更新权重
4. calc_delta：反向传播核心，先算输出层delta，再倒序算各隐藏层delta
5. update_weight：遍历全部连接更新权重
6. predict：前向传播，返回输出层节点输出
7. get_gradient：获取梯度，用于梯度检查
8. dump：打印全部网络调试信息

思考：
训练使用随机梯度下降SGD，每读完一个样本就执行一次反向传播更新权重，不是等全部样本算完再更新。

In [5]:
class Network(object):
    def __init__(self, layers):
        self.connections = Connections()
        self.layers = []
        layer_count = len(layers)
        for i in range(layer_count):
            self.layers.append(Layer(i, layers[i]))
        for layer in range(layer_count -1):
            connections = [Connection(upstream_node, downstream_node)
                           for upstream_node in self.layers[layer].nodes
                           for downstream_node in self.layers[layer+1].nodes[:-1]]
            for conn in connections:
                self.connections.add_connection(conn)
                conn.downstream_node.append_upstream_connection(conn)
                conn.upstream_node.append_downstream_connection(conn)

    def train(self, labels, data_set, rate, iteration):
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d], data_set[d], rate)

    def train_one_sample(self, label, sample, rate):
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)

    def calc_delta(self, label):
        output_nodes = self.layers[-1].nodes
        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])
        for layer in self.layers[-2::-1]:
            for node in layer.nodes[:-1]:
                node.calc_hidden_layer_delta()

    def update_weight(self, rate):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)

    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()

    def predict(self, sample):
        self.layers[0].set_output(sample)
        for i in range(1, len(self.layers)):
            self.layers[i].calc_output()
        return list(map(lambda node: node.output, self.layers[-1].nodes[:-1]))

    def dump(self):
        for layer in self.layers:
            layer.dump()

梯度检查工具函数
梯度检查用途：验证反向传播手写的梯度计算是否正确。
原理：导数定义，用微小扰动epsilon前后的损失，计算数值梯度；和反向传播算出解析梯度对比，如果几乎相等，代表反向传播代码写的没问题。

函数network_error：计算两个向量平方误差总和。
gradient_check：输入网络、样本特征、样本标签，循环每一条连接，对比数值梯度与解析梯度。

思考：
手写反向传播极容易出错，肉眼很难排查。梯度检查是调试神经网络非常重要的手段。注意epsilon取值要很小，一般取1e‑4。

In [6]:
def gradient_check(network, sample_feature, sample_label):
    network_error = lambda vec1, vec2: \
        0.5 * reduce(lambda a, b: a + b,
        map(lambda v: (v[0] - v[1]) * (v[0] - v[1]),
        zip(vec1, vec2)))

    network.get_gradient(sample_feature, sample_label)
    for conn in network.connections.connections:
        actual_gradient = conn.get_gradient()
        epsilon = 0.0001
        conn.weight += epsilon
        error1 = network_error(network.predict(sample_feature), sample_label)
        conn.weight -= 2 * epsilon
        error2 = network_error(network.predict(sample_feature), sample_label)
        expected_gradient = (error2 - error1) / (2 * epsilon)

        print('expected gradient: \t%f\nactual gradient: \t%f' % (
        expected_gradient, actual_gradient))

# 7. 实战：MNIST手写数字识别
### 任务介绍
MNIST手写数字数据集：
- 训练集60000张28×28灰度手写数字图片，数字0‑9
- 测试集10000张图片，用来评估模型泛化错误率
- 任务：10分类任务，输入一张图片，识别是0~9哪一个数字

### 超参数设计
1. 输入层：28*28=784个节点，每一个像素点对应一个输入节点。像素0‑255。
2. 输出层：10个节点，分别对应数字0~9。
3. 标签处理：one‑hot编码；文档中不使用0/1，使用平滑标签：正确类别输出0.9，其余类别输出0.1。
例如数字2，标签向量为 `[0.1,0.1,0.9,0.1,0.1,0.1,0.1,0.1,0.1,0.1]`
> 使用0.1、0.9而不是0、1，是为了避开sigmoid函数输出0/1区域，该区域梯度几乎为0，会造成梯度消失，参数很难更新。
4. 隐藏层节点：文档经验选择300个节点。
网络结构：`layers = [784, 300, 10]`

思考:
1. 超参数（层数、每层节点数）没有严格数学公式，依靠经验调参。
2. 全连接网络参数量巨大，本网络参数总数约23万。参数量大，训练集不足时容易发生过拟合。
3. 训练策略：每训练若干轮，使用测试集评估错误率；当测试集错误率开始上升，代表出现过拟合，停止训练。


In [21]:
import struct
import time

class Loader(object):
    def __init__(self, path, count):
        self.path = path
        self.count = count

    def get_file_content(self):
        f = open(self.path, 'rb')
        content = f.read()
        f.close()
        return content

    def to_int(self, byte):
        return struct.unpack('B', byte)[0]

class ImageLoader(Loader):
    def get_picture(self, content, index):
        start = index * 28 * 28 + 16
        picture = []
        for i in range(28):
            picture.append([])
            for j in range(28):
                picture[i].append(
                    self.to_int(content[start + i * 28 + j]))
        return picture

    def get_one_sample(self, picture):
        sample = []
        for i in range(28):
            for j in range(28):
                sample.append(picture[i][j])
        return sample

    def load(self):
        content = self.get_file_content()
        data_set = []
        for index in range(self.count):
            data_set.append(
                self.get_one_sample(
                    self.get_picture(content, index)))
        return data_set

class LabelLoader(Loader):
    def load(self):
        content = self.get_file_content()
        labels = []
        for index in range(self.count):
            labels.append(self.norm(content[index + 8]))
        return labels

    def norm(self, label):
        label_vec = []
        label_value = self.to_int(label)
        for i in range(10):
            if i == label_value:
                label_vec.append(0.9)
            else:
                label_vec.append(0.1)
        return label_vec

def get_training_data_set():
    image_loader = ImageLoader(r"D:\crd_learning\deep_learning\data\train-images.idx3-ubyte", 60000)
    label_loader = LabelLoader(r"D:\crd_learning\deep_learning\data\train-labels.idx1-ubyte", 60000)
    return image_loader.load(), label_loader.load()

def get_test_data_set():
    image_loader = ImageLoader(r"D:\crd_learning\deep_learning\data\t10k-images.idx3-ubyte", 10000)
    label_loader = LabelLoader(r"D:\crd_learning\deep_learning\data\t10k-labels.idx1-ubyte", 10000)
    return image_loader.load(), label_loader.load()

def get_result(vec):
    max_value_index = 0
    max_value = 0
    for i in range(len(vec)):
        if vec[i] > max_value:
            max_value = vec[i]
            max_value_index = i
    return max_value_index

def evaluate(network, test_data_set, test_labels):
    error = 0
    total = len(test_data_set)
    for i in range(total):
        label = get_result(test_labels[i])
        predict = get_result(network.predict(test_data_set[i]))
        if label != predict:
            error +=1
    return float(error)/float(total)

def now():
    return time.strftime('%Y‑%m‑%d %H:%M:%S',time.localtime())

def train_and_evaluate():
    last_error_ratio = 1.0
    epoch = 0
    train_data_set, train_labels = get_training_data_set()
    test_data_set, test_labels = get_test_data_set()
    network = Network([784, 300, 10])
    while True:
        epoch += 1
        network.train(train_labels, train_data_set, 0.3, 1)
        print('%s epoch %d finished' % (now(), epoch))
        if epoch % 10 == 0:
            error_ratio = evaluate(network, test_data_set, test_labels)
            print('%s after epoch %d, error ratio is %f' % (now(), epoch, error_ratio))
            if error_ratio > last_error_ratio:
                break
            else:
                last_error_ratio = error_ratio


#if __name__ == '__main__':
  #  train_and_evaluate()